# ASQs data

In [0]:
SELECT *
FROM main.field_sts_metrics.silver_shared_stsasq;

In [0]:
SELECT DISTINCT(status)
FROM main.field_sts_metrics.silver_shared_stsasq;

In [0]:
SELECT * 
FROM main.field_sts_metrics.gold_shared_asqrelativemetrics;

# Features Data

In [0]:

SELECT *
FROM main.data_product_features.account_metrics;


In [0]:
SELECT 
  a.*,
  b.date as usage_date,
  b.lakeflow_dbus
FROM main.field_sts_metrics.silver_shared_stsasq a
LEFT JOIN main.data_product_features.account_metrics b
  ON a.account_id = b.customer_id
ORDER BY a.account_name, b.date DESC;

In [0]:
WITH asq_data as (

SELECT
      make_date(s.created_year, s.created_month, s.created_day) as created_date,
      s.created_day,
      s.created_month,
      s.created_year,
      s.account_name,
      s.fiscal_quarter,
      s.fiscal_quarter_asq_startdate,
      s.account_id,
      s.asq_name,
      s.owner_name,
      s.created_by_name,
      s.start_date,
      s.end_date,
      s.account_bu,
      s.sales_subregion_l1,
      s.sales_subregion_l2,
      s.sales_subregion_l3,
      s.owner_role,
      s.status,
      s.support_type,
      s.additional_services,
      s.asq_url,
      s.ts,
      m.months_after_start,
      year(add_months(to_date(s.start_date), m.months_after_start)) AS year_month_of_consumption,
      month(add_months(to_date(s.start_date), m.months_after_start)) AS month_of_consumption
    FROM
      field_sts_metrics.silver_shared_stsasq s
    LATERAL VIEW
      EXPLODE(SEQUENCE(-6, 12)) m AS months_after_start
  ),

WHERE status != 'Rejected,'



features_data as (

  SELECT date,
  customer_id,
  lakeflow_dbus
  
  FROM main.data_product_features.account_metrics
)

SELECT 
  CONCAT(asq.account_name," / ",asq.asq_name) as account_asq,
  asq.*,
  f.date as usage_date,
  f.lakeflow_dbus
FROM asq_data asq
LEFT JOIN features_data f
  ON asq.account_id = f.customer_id
ORDER BY asq.account_name, f.date DESC

